In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import timm
from tqdm import tqdm
from pathlib import Path

In [ ]:
# 1. CONFIGURATION
class CFG:
    SR = 32000
    DURATION = 5
    CHUNK_SIZE = SR * DURATION
    INPUT_DIR = Path("/kaggle/input/competitions/birdclef-2026/test_soundscapes/")
   
   
    MODEL_PATH_B3_RARE_AMPH = "/kaggle/input/models/gany24558/birdclef-apr-20-2026-gc/pytorch/rare-amphibian-oversample/1/bird_model_best.pth"
    MODEL_PATH_B3_PSEUDO = "/kaggle/input/models/gany24558/birdclef-apr-20-2026-gc/pytorch/rare-amphibian-oversample/2/bird_model_best.pth"
    
    """
    MODEL_PATH_B0 = "/kaggle/input/models/gany24558/birdclef-apr-20-2026-gc/pytorch/syntheticdata/8/bird_model_best.pth"
    MODEL_PATH_AMPH_ONLY_B0 = "/kaggle/input/models/gany24558/birdclef-apr-20-2026-gc/pytorch/amphibian-specific-b0/1/bird_model_best.pth"    
    """

   
    NUM_CLASSES = 234
    THRESHOLD = 0.00
    B3_COMP_RARE_AMPH = 0.70
    B3_PSEUDO_LABELED = 0.30
    """
    B0_WEIGHT = 0.10  # B0 is also secondary
    B0_AMPH_ONLY_WEIGHT = 0.05    
    """


for dirname, _, filenames in os.walk('/kaggle/input/competitions/birdclef-2026/test_soundscapes/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# REFINED MODEL CLASS
class BirdModel(nn.Module):
    def __init__(self, model_name='efficientnet_b3', num_classes=234):
        super().__init__()
        # 1. Create model with 1 input channel instead of 3
        self.backbone = timm.create_model(model_name, pretrained=False, in_chans=1)
        
        # 2. Based on your error message, your trained weights use "fc" 
        # instead of "backbone.classifier". Let's map it:
        self.fc = nn.Linear(self.backbone.classifier.in_features, num_classes)
        
        # 3. Disable the default classifier so it doesn't conflict
        self.backbone.classifier = nn.Identity()
        
    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x) # Use the "fc" layer as in your training
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Load B3 Model trained on composite primary labels and no soundscape mixing (234 classes)
model_b3_rare_amph = BirdModel(model_name='efficientnet_b3', num_classes=234)
model_b3_rare_amph.load_state_dict(torch.load(CFG.MODEL_PATH_B3_RARE_AMPH, map_location=device))
model_b3_rare_amph.to(device).eval()


# Load B3 Model trained on composite primary labels and no soundscape mixing (234 classes)
model_b3_pseudo = BirdModel(model_name='efficientnet_b3', num_classes=234)
model_b3_pseudo.load_state_dict(torch.load(CFG.MODEL_PATH_B3_PSEUDO, map_location=device))
model_b3_pseudo.to(device).eval()

"""
# Load B0 Model (206 classes)
model_b0 = BirdModel(model_name='efficientnet_b0', num_classes=206)
model_b0.load_state_dict(torch.load(CFG.MODEL_PATH_B0, map_location=device))
model_b0.to(device).eval()

# Load B0 Model trained on amphibians only (234 classes)
model_b0_amph_only = BirdModel(model_name='efficientnet_b0', num_classes=234)
model_b0_amph_only.load_state_dict(torch.load(CFG.MODEL_PATH_AMPH_ONLY_B0, map_location=device))
model_b0_amph_only.to(device).eval()

"""




In [ ]:
# 4. LOAD SUBMISSION FORMAT + CREATE SPECIES MAPPING

# Load the official sample submission to get the 234 required species columns
sample_sub_path = Path("/kaggle/input/competitions/birdclef-2026/sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path)
target_species = sample_sub.columns[1:].tolist()

In [ ]:
# Based on your sidebar screenshot, the data is here:
data_root = Path("/kaggle/input/competitions/birdclef-2026")

In [ ]:
# Load full 234 taxonomy
taxonomy_path = Path("/kaggle/input/competitions/birdclef-2026/taxonomy.csv")
taxonomy_df = pd.read_csv(taxonomy_path)
all_species_234 = sorted(taxonomy_df['primary_label'].unique())
mapping_234 = {s: i for i, s in enumerate(all_species_234)}

print(f"✅ Mapping created for {len(mapping_234)} species.")
# Verification check:
if len(mapping_234) != 234:
    print(f"⚠️ Warning: Found {len(mapping_234)} species, expected 234!")

# Load the 206 training labels used for B0
# Assuming you have the train.csv available
train_df = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train.csv")
species_206 = sorted(train_df['primary_label'].unique())

# Create an index map: which index in B3 (234) corresponds to the index in B0 (206)?
b0_to_b3_idx = [mapping_234[s] for s in species_206]

In [ ]:
def get_spectrogram(audio_chunk):
    # Matches the 1-channel logic your model expects
    S = librosa.feature.melspectrogram(y=audio_chunk, sr=CFG.SR, n_mels=128, fmax=16000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    return S_dB

In [ ]:
def get_ensemble_probs(file_path):
    y, _ = librosa.load(file_path, sr=CFG.SR)
    ensemble_probs = []
   
    for i in range(0, len(y), CFG.CHUNK_SIZE):
        chunk = y[i:i + CFG.CHUNK_SIZE]
        if len(chunk) < CFG.CHUNK_SIZE:
            chunk = np.pad(chunk, (0, CFG.CHUNK_SIZE - len(chunk)))
       
        spec = get_spectrogram(chunk)
        tensor_spec = torch.from_numpy(spec)[None, None, ...].float().to(device)
       
        with torch.no_grad():
            # 0. Get B3 with Composite Primary Labels (Full 234)
            out_b3_rare_amph = model_b3_rare_amph(tensor_spec)
            probs_b3_rare_amph = torch.sigmoid(out_b3_rare_amph).cpu().numpy()[0]

            # 0. Get B3 with Composite Primary Labels (Full 234)
            out_b3_pseudo = model_b3_pseudo(tensor_spec)
            probs_b3_pseudo = torch.sigmoid(out_b3_pseudo).cpu().numpy()[0]            

            """
            # 2. Get B0 Probs (206)
            out_b0 = model_b0(tensor_spec)
            probs_b0_raw = torch.sigmoid(out_b0).cpu().numpy()[0]

            # 3. Get Amphibian only B0 Probs (234)
            out_b0_amph_only = model_b0_amph_only(tensor_spec)
            probs_b0_amph_only = torch.sigmoid(out_b0_amph_only).cpu().numpy()[0]
           
            # 3. Align B0 to 234 classes
            probs_b0_aligned = np.zeros(234, dtype=np.float32)
            for old_idx, new_idx in enumerate(b0_to_b3_idx):
                probs_b0_aligned[new_idx] = probs_b0_raw[old_idx]                     
            """
   


            blended_probs = (probs_b3_rare_amph * CFG.B3_COMP_RARE_AMPH) + (probs_b3_pseudo * CFG.B3_PSEUDO_LABELED)
            ensemble_probs.append(blended_probs)
           
    del y
    return np.array(ensemble_probs)

In [ ]:
def aggregate_and_format(
    raw_probs,
    file_id,
    target_species,
    species_mapping
):
    """
    Recall-oriented temporal smoothing + probability sharpening.

    Key ideas:
    --------------------------------
    1. Preserve weak detections
       (BirdCLEF rewards recall heavily)

    2. Use max-based smoothing
       (validated by leaderboard)

    3. Apply mild probability sharpening
       because model appears underconfident

    4. Keep submission-compatible formatting
    """

    raw_probs = raw_probs.astype(np.float32)

    num_chunks = len(raw_probs)

    # ------------------------------------------------------------
    # TEMPORAL SMOOTHING
    # ------------------------------------------------------------
    smoothed_probs = np.zeros_like(raw_probs)

    for i in range(num_chunks):

        # Slightly wider context window
        start = max(0, i - 2)
        end   = min(num_chunks, i + 3)

        window = raw_probs[start:end]

        # --------------------------------------------------------
        # Recall-oriented smoothing
        #
        # Max preserves faint / partial calls
        # Mean reduces extreme instability slightly
        # --------------------------------------------------------
        max_probs  = np.max(window, axis=0)
        mean_probs = np.mean(window, axis=0)

        smoothed_probs[i] = (
            0.85 * max_probs +
            0.15 * mean_probs
        )

    # ------------------------------------------------------------
    # PROBABILITY SHARPENING
    #
    # Your diagnostics suggest:
    # - AUC extremely high
    # - AP decent
    # - probabilities underconfident
    #
    # Power transform increases confidence separation
    # while mostly preserving ranking.
    # ------------------------------------------------------------
    smoothed_probs = np.power(smoothed_probs, 0.75)

    # ------------------------------------------------------------
    # FORMATTING
    # ------------------------------------------------------------
    file_results = []

    for i in range(num_chunks):

        row_id = f"{file_id}_{(i + 1) * 5}"

        pred_dict = {
            "row_id": row_id
        }

        chunk_probs = smoothed_probs[i]

        """
        # Optional geographic suppression
        chunk_probs = apply_geographic_suppression(
            chunk_probs,
            species_mapping
        )
        """

        # --------------------------------------------------------
        # Thresholding
        # --------------------------------------------------------
        chunk_probs = np.where(
            chunk_probs > CFG.THRESHOLD,
            chunk_probs,
            0.0
        )

        # --------------------------------------------------------
        # Populate submission row
        # --------------------------------------------------------
        for species in target_species:

            if species in species_mapping:

                pred_dict[species] = chunk_probs[
                    species_mapping[species]
                ]

            else:

                pred_dict[species] = 0.0

        file_results.append(pred_dict)

    return file_results

In [ ]:
def predict_file(file_path):
    """
    Main entry point used by the loop.
    """
    # 1. Get raw numbers from the model
    raw_probs = get_ensemble_probs(file_path)
    
    # 2. Smooth and format
    results = aggregate_and_format(
        raw_probs, 
        file_path.stem, 
        target_species, 
        mapping_234
    )
    
    gc.collect()
    return results

In [ ]:

# 6. MAIN SUBMISSION LOOP
# Check for hidden test files; fallback to train files for the 'Commit' run
test_dir = Path("/kaggle/input/competitions/birdclef-2026/test_soundscapes")
test_files = list(test_dir.glob("*.ogg"))

if len(test_files) == 0:
    print("No test files found. Running on subset of training data for verification.")
    test_dir = Path("/kaggle/input/competitions/birdclef-2026/train_soundscapes")
    test_files = list(test_dir.glob("*.ogg"))[:3]

all_preds = []
for file in tqdm(test_files):
    all_preds.extend(predict_file(file))

# 7. GENERATE SUBMISSION FILE
submission_df = pd.DataFrame(all_preds)
# Ensure exact Kaggle column ordering
submission_df = submission_df[["row_id"] + target_species]
# Replace NaNs if any appear
submission_df = submission_df.fillna(0.0)
submission_df.to_csv("submission.csv", index=False)
